<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Modern AI: LLMs & Generative AI** · Week 1 · How Large Language Models Work · Lesson 6 of 24

# Week 1 Project: Train and Study Your Own Language Model

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

**This lesson uses a GPU.** Before you start: **Runtime → Change runtime type → T4 GPU → Save**. The code also runs without a GPU, only more slowly.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative).

**Learning objective:** Train a small GPT on a text you choose, measure it with validation loss, explore sampling settings, and explain its strengths and failures — including hallucination — in a short report.

## The brief

A teacher wants a demonstration for a school club: "How does ChatGPT write?" Your job is to train a small language model, show how it improves, show what sampling does, and explain honestly what it gets wrong.

## Part 1 — Choose your text

Use the market sentences, or replace `text` with your own: proverbs, simple stories or poems you wrote yourself, or public-domain text. Aim for at least 50,000 characters of fairly consistent style. Do not use private messages or copyrighted books.

In [ ]:
import random

random.seed(0)
names = ["Aline", "Eric", "Grace", "Jean", "Diane", "Patrick", "Claudine", "Samuel", "Ange", "Didier"]
places = ["Kigali", "Huye", "Musanze", "Rubavu", "Nyagatare", "Rusizi", "Muhanga"]
goods = ["tomatoes", "beans", "bananas", "avocados", "maize", "potatoes", "milk", "coffee", "rice", "cassava"]
verbs = ["sells", "buys", "carries", "cooks", "packs", "counts"]
times = ["in the morning", "at noon", "in the evening", "on Saturday", "after the rain", "before school"]
moods = ["happy", "tired", "busy", "proud", "calm", "hungry"]
templates = [
    "{n} {v} {g} in {p} {t}.",
    "In {p}, {n} {v} {g} {t}.",
    "{n} is {f} because the {g} are fresh.",
    "{t}, {n} goes to the market in {p}.",
    "{n} pays for the {g} with mobile money.",
    "The price of {g} in {p} is low {t}.",
    "{n} tells {m} that the {g} from {p} are the best.",
    "{n} and {m} share {g} {t}.",
]

def sentence():
    n, m = random.sample(names, 2)
    s = random.choice(templates).format(n=n, m=m, v=random.choice(verbs), g=random.choice(goods),
                                        p=random.choice(places), t=random.choice(times), f=random.choice(moods))
    return s[0].upper() + s[1:]

text = "\n".join(sentence() for _ in range(4000))
print(len(text), "characters")
print(text[:250])

## Part 2 — Build the model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

BLOCK = 64          # how many characters the model can look back at (its context window)

def get_batch(source, batch_size=32):
    starts = torch.randint(len(source) - BLOCK - 1, (batch_size,))
    x = torch.stack([source[s:s + BLOCK] for s in starts])
    y = torch.stack([source[s + 1:s + BLOCK + 1] for s in starts])   # the next character at every position
    return x.to(device), y.to(device)

class Block(nn.Module):
    # One transformer block: masked self-attention + feed-forward, with residuals.
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        mask = torch.triu(torch.ones(BLOCK, BLOCK, dtype=torch.bool), diagonal=1)
        self.register_buffer("mask", mask)              # True = may NOT look (the future)

    def forward(self, x):
        T = x.shape[1]
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=self.mask[:T, :T], need_weights=False)
        x = x + a
        return x + self.ff(self.ln2(x))

class TinyGPT(nn.Module):
    def __init__(self, vocab, d=96, heads=4, layers=3):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(BLOCK, d)
        self.blocks = nn.Sequential(*[Block(d, heads) for _ in range(layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)

    def forward(self, idx):
        T = idx.shape[1]
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        return self.head(self.ln(self.blocks(x)))       # logits: a score for every next character

    @torch.no_grad()
    def generate(self, idx, n_new, temperature=1.0, top_k=None):
        for _ in range(n_new):
            logits = self(idx[:, -BLOCK:])[:, -1, :] / temperature
            if top_k is not None:
                cutoff = torch.topk(logits, top_k).values[:, [-1]]
                logits[logits < cutoff] = -float("inf")
            next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

@torch.no_grad()
def estimate_loss(model, steps=20):
    model.eval()
    out = {}
    for name, source in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(steps):
            x, y = get_batch(source)
            logits = model(x)
            losses.append(F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1)).item())
        out[name] = sum(losses) / len(losses)
    model.train()
    return out

def train_gpt(steps=1500, seed=0):
    torch.manual_seed(seed)
    model = TinyGPT(len(chars)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for step in range(steps + 1):
        if step % 500 == 0:
            l = estimate_loss(model)
            print(f"step {step:4d}: train loss {l['train']:.3f}, validation loss {l['val']:.3f}")
        if step == steps:
            break
        x, y = get_batch(train_data)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model

print("vocabulary:", len(chars), "characters")

## Part 3 — Train and measure

In [ ]:
model = train_gpt(steps=1500)
final = estimate_loss(model, steps=50)
print(f"final: train {final['train']:.3f}, validation {final['val']:.3f}")

**Validation loss** is measured on the last 10% of the text, which the model never trains on. If validation loss rises while training loss falls, the model is memorising (overfitting) — exactly as in earlier courses.

**Perplexity** = e^loss is a common way to report it: roughly "how many characters the model is choosing between". A loss of 1.0 means a perplexity of about 2.7.

In [ ]:
import math
print("validation perplexity:", round(math.exp(final["val"]), 2))

## Part 4 — Explore sampling

In [ ]:
for T in [0.3, 0.8, 1.3]:
    torch.manual_seed(2)
    out = model.generate(torch.tensor([encode("In ")], device=device), 150, temperature=T)
    print(f"--- temperature {T} ---")
    print(decode(out[0].tolist()))

Record what changes: repetition at low temperature, spelling mistakes and odd words at high temperature.

## Part 5 — Test for "hallucination"

In [ ]:
torch.manual_seed(3)
samples = decode(model.generate(torch.tensor([encode("\n")], device=device), 2000, temperature=0.8)[0].tolist())
generated = [s.strip() for s in samples.split("\n") if s.strip().endswith(".")]
training_sentences = set(text.split("\n"))
new = [s for s in generated if s not in training_sentences]
print(f"{len(generated)} complete sentences, {len(new)} never appeared in the training text")
print("examples of new sentences:", new[:3])

Many generated sentences are **new combinations**: grammatical, believable, and never in the data. In our market text that is harmless. In a medical or legal assistant, a fluent new "fact" can be dangerous. This is why real systems ground answers in documents (RAG, Week 3) and why people must check important outputs.

## Part 6 — Report (one page)

1. **Data:** what text, how many characters, how many different characters.
2. **Model:** layers, width, context length, parameter count.
3. **Training:** loss curve numbers at steps 0, 500, 1000, 1500; final validation perplexity.
4. **Sampling:** three short samples at different temperatures with one sentence each on what you see.
5. **Hallucination:** how many new sentences, two examples, and why this matters for real LLMs.
6. **From tiny to large:** three differences between your model and a commercial LLM.

## Marking guide (self-check)

| Area | Excellent work |
|---|---|
| Training | Model trains; train and validation losses reported |
| Understanding | Causal mask, next-token targets and sampling explained in your own words |
| Experiment | Temperature comparison with evidence |
| Critical thinking | Hallucination measured and explained |
| Communication | A school club member understands the report |

**Takeaway:** You trained a language model from zero and saw both its power (fluent patterns) and its core weakness (likely is not the same as true).

## Try it yourself

Complete Parts 1–6. Challenge: double `BLOCK` to 128 and the width to 128, train again on a GPU, and compare validation loss and samples.

---
Finished? Go back to [Modern AI: LLMs & Generative AI on WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*